In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

housing = pd.read_csv("housing.csv")

# Checking linearity 
# housing.plot.scatter(x="Enter feature name", y="Price", alpha=0.4) 

# There's a linear relationship between the property price and all its features, except for the avg. number of bedrooms

decisions = {
    "1": "Avg. Area Income",
    "2": "Avg. Area House Age",
    "3": "Avg. Area Number of Rooms",
    "4": "Area Population",
    "5": "Use all of the features above"
}

all_features = [
    "Avg. Area Income",
    "Avg. Area House Age",
    "Avg. Area Number of Rooms",
    "Area Population"
]

default_values = {
    "Avg. Area Income": [50000, 60000, 70000],
    "Avg. Area House Age": [5, 10, 15],
    "Avg. Area Number of Rooms": [6, 7, 8],
    "Area Population": [30000, 50000, 80000]
}

print("Available features for price prediction:")
for key, feature in decisions.items():
    print(f"{key}. {feature}")
print()

choice = input("Select a feature (1-5): ").strip()

if choice not in decisions:
    print("Invalid choice. Please run again.")
else:
    selected_feature = decisions[choice]

    if choice == "5":
        formula = "Price ~ " + " + ".join([f'Q("{feature}")' for feature in all_features])
        predict_features = all_features
    else:
        formula = f'Price ~ Q("{selected_feature}")'
        predict_features = [selected_feature]

    lm = smf.ols(formula, data=housing).fit()

    print(f"\nModel Summary for {selected_feature}:")
    print(lm.summary())

    if choice == "5":
        print("Enter values for all features. Leave any field blank to use default sample values for that feature.")
        sample_values = {}

        for feature in all_features:
            value = input(f"Enter a value for '{feature}': ").strip()
            if value == "":
                sample_values[feature] = [default_values[feature][0]]
                print(f"Using default value {default_values[feature][0]} for '{feature}'.")
            else:
                sample_values[feature] = [float(value)]


    else:
        value = input(f"Enter a value for '{selected_feature}' to predict price: ").strip()
        if value == "":
            sample_values = default_values.copy()
            print("Using default values for prediction.")
        else:
            sample_values = {selected_feature: [float(value)]}

    predictions = pd.DataFrame({feature: sample_values[feature] for feature in predict_features})
    predictions["Predicted Price"] = lm.predict(predictions)

    print(f"\nSample Predictions for {selected_feature}:")
    print(predictions)

    if choice == "5":
        plt.scatter(housing.index, housing["Price"], alpha=0.4, label="Actual Price")
        plt.scatter(predictions.index, predictions["Predicted Price"], color='red', s=150, marker='*', label="Predictions")
        plt.xlabel("Sample Index")
        plt.ylabel("Price")
        plt.title("Price Prediction using all features")
    else:
        plt.scatter(housing[selected_feature], housing["Price"], alpha=0.4, label="Training Data")
        plt.scatter(predictions[selected_feature], predictions["Predicted Price"], color='red', s=150, marker='*', label="Predictions")
        plt.xlabel(selected_feature)
        plt.ylabel("Price")
        plt.title(f"Price Prediction based on {selected_feature}")

    plt.legend()
    plt.show()